In [1]:
# =============================================================================
# Q&A EXTRACTOR - MULTI-PROVIDER PARALLEL PROCESSING
# =============================================================================
# This script extracts Question & Answer pairs from dawah conversations
# using parallel processing. Supports both OpenAI and Google Gemini models.
# =============================================================================


# =============================================================================
# CELL 1: Installation
# =============================================================================
# Run this cell first (uncomment if needed)

# !pip install openai pandas json-repair tqdm python-dotenv google-generativeai


# =============================================================================
# CELL 2: Imports
# =============================================================================

import json
import os
import pandas as pd
from json_repair import repair_json
from tqdm import tqdm
import warnings
import ast
from datetime import datetime
import statistics
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from google.genai import types as genai_types
import concurrent.futures
import threading
import time

warnings.filterwarnings("ignore")

# Load environment variables from .env file (if exists)
load_dotenv()

print("All imports successful!")



All imports successful!


In [2]:

# =============================================================================
# CELL 3: Configuration
# =============================================================================

# ============================================
# PROVIDER & MODEL SETTINGS (from .env)
# ============================================
PROVIDER = os.environ.get("LLM_PROVIDER", "openai").lower()   # "openai" or "gemini"
MODEL_ID = os.environ.get("LLM_MODEL", "gpt-4o-mini")

# ============================================
# MODEL PRICING (per 1M tokens, as of April 2026)
# ============================================
MODEL_PRICING = {
    # ===================== OpenAI =====================
    "gpt-4o-mini":       {"input": 0.15,  "output": 0.60},
    "gpt-5":             {"input": 2.00,  "output": 15.00},
    "gpt-5-mini":        {"input": 0.25,  "output": 0.60},
    "gpt-5-nano":        {"input": 0.10,  "output": 0.40},
    "o3":                {"input": 2.00,  "output": 8.00},
    "o3-mini":           {"input": 1.10,  "output": 4.40},
    # ===================== Gemini 3.x (Latest) =====================
    # Gemini 3.1 Pro — Flagship, complex reasoning, agentic & coding (Preview)
    "gemini-3.1-pro-preview":            {"input": 2.00, "output": 12.00},
    # Gemini 3 Flash — Frontier performance at fraction of cost (Preview)
    "gemini-3-flash-preview":            {"input": 0.50, "output": 3.00},
    # Gemini 3.1 Flash-Lite — Most cost-efficient, high-volume tasks (Preview)
    "gemini-3.1-flash-lite-preview":     {"input": 0.25, "output": 1.50},

    # ===================== Gemini 2.5 (Stable) =====================
    # Gemini 2.5 Pro — Advanced reasoning & coding, 1M context
    "gemini-2.5-pro":                    {"input": 1.25, "output": 10.00},
    # Gemini 2.5 Flash — Price-performance, hybrid reasoning, 1M context
    "gemini-2.5-flash":                  {"input": 0.30, "output": 2.50},
    # Gemini 2.5 Flash-Lite — Smallest, most cost-effective in 2.5 family
    "gemini-2.5-flash-lite":             {"input": 0.10, "output": 0.40},
    # Gemini 2.5 Flash-Lite Preview (latest optimization)
    "gemini-2.5-flash-lite-preview-09-2025": {"input": 0.10, "output": 0.40},

    # ===================== Gemini 2.0 (Deprecated) =====================
    # Gemini 2.0 Flash — Previous gen workhorse (deprecated, use 2.5+)
    "gemini-2.0-flash":                  {"input": 0.10, "output": 0.40},
    # Gemini 2.0 Flash-Lite — Previous gen fastest (deprecated)
    "gemini-2.0-flash-lite":             {"input": 0.075, "output": 0.30},
}

# ============================================
# PROCESSING MODE
# ============================================
PROCESSING_MODE = os.environ.get("PROCESSING_MODE", "parallel").lower()  # "parallel" or "batch"
# - "parallel": Real-time processing with ThreadPoolExecutor (current behavior)
# - "batch":    Gemini Batch API — 50% cost, async (submit & poll, up to 24h)

# ============================================
# PARALLEL PROCESSING SETTINGS
# ============================================
NUM_WORKERS = 16  # Number of parallel workers (recommended: 5-20)
                  # Adjust based on your API rate limits:
                  # - Tier 1: Use 5-10 workers
                  # - Tier 2+: Use 10-20 workers

# ============================================
# FILE PATHS - UPDATE THESE
# ============================================
INPUT_FILE = r"S:\Midade work\Islam chat conversation analysis\Conversation question extraction\organized_conversations_with_language.csv"
OUTPUT_FILE = f"Islam_chat_questions_extraction.csv"

# ============================================
# CHECKPOINTING SETTINGS
# ============================================
CHECKPOINT_FILE = f"qa_checkpoint_{MODEL_ID.replace('.', '_').replace('-', '_')}.json"
CHECKPOINT_EVERY = 10  # Save progress every N completed rows

# ============================================
# BATCH MODE SETTINGS
# ============================================
BATCH_JOB_FILE = f"batch_job_{MODEL_ID.replace('.', '_').replace('-', '_')}.json"  # Stores job name for resume
BATCH_JSONL_FILE = "batch_requests.jsonl"  # Temp JSONL file for batch input
BATCH_POLL_INTERVAL = 30  # Seconds between status polls

# ============================================
# GENERATION SETTINGS
# ============================================
MAX_TOKENS = 8192  # Increased — GPT-5 models need more room for verbose Q&A JSON output
TEMPERATURE = 0.15  # Low temperature for consistent extraction

# ============================================
# RETRY SETTINGS
# ============================================
MAX_RETRIES = 5
RETRY_BASE_DELAY = 1  # Base delay in seconds for exponential backoff

# ============================================
# VALIDATION
# ============================================
if MODEL_ID not in MODEL_PRICING:
    print(f"⚠️  WARNING: Model '{MODEL_ID}' not found in MODEL_PRICING dictionary.")
    print(f"   Cost estimation will show $0. Add pricing to MODEL_PRICING if needed.")

# Validate batch mode requires Gemini
if PROCESSING_MODE == "batch" and PROVIDER != "gemini":
    raise ValueError(
        f"Batch mode only supports Gemini provider, but LLM_PROVIDER='{PROVIDER}'.\n"
        f"Either set LLM_PROVIDER=gemini or PROCESSING_MODE=parallel."
    )

print(f"Provider: {PROVIDER.upper()}")
print(f"Selected Model: {MODEL_ID}")
print(f"Processing Mode: {PROCESSING_MODE.upper()}")
if PROCESSING_MODE == "parallel":
    print(f"Parallel Workers: {NUM_WORKERS}")
print(f"Input File: {INPUT_FILE}")
print(f"Output File: {OUTPUT_FILE}")
print(f"Checkpoint File: {CHECKPOINT_FILE}")



Provider: GEMINI
Selected Model: gemini-3.1-flash-lite-preview
Processing Mode: BATCH
Input File: S:\Midade work\Islam chat conversation analysis\Conversation question extraction\organized_conversations_with_language.csv
Output File: Islam_chat_questions_extraction.csv
Checkpoint File: qa_checkpoint_gemini_3_1_flash_lite_preview.json


In [3]:

# =============================================================================
# CELL 4: Initialize LLM Client (Provider-Aware)
# =============================================================================

if PROVIDER == "openai":
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY not found. Set it in .env or environment variables.")
    client = OpenAI(api_key=api_key)
    print(f"✅ OpenAI client initialized! Model: {MODEL_ID}")

elif PROVIDER == "gemini":
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("GOOGLE_API_KEY not found. Set it in .env or environment variables.")
    client = genai.Client(api_key=api_key)
    print(f"✅ Gemini client initialized (google-genai SDK)! Model: {MODEL_ID}")

else:
    raise ValueError(f"Unknown provider '{PROVIDER}'. Use 'openai' or 'gemini'.")



Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Gemini client initialized (google-genai SDK)! Model: gemini-3.1-flash-lite-preview


In [4]:

# =============================================================================
# CELL 5: Q&A Extraction Prompt
# =============================================================================

EXTRACT_SYSTEM_PROMPT = """You are a Q&A extraction and translation specialist for an Islamic dawah (missionary) knowledge base.
Your task is to extract question-and-answer pairs from conversations between a Dawah Bot ("AI") and human users ("User"), and transform them into polished, standalone FAQ entries.
The bot is designed to promote Islam, answer religious questions, and engage users in dialogue about faith.
ALL output must be in English — accurately translate any non-English content to English.
Output strict JSON only. No markdown, no commentary."""

EXTRACT_USER_PROMPT_TEMPLATE = """### TASK:
Extract all question-and-answer pairs from this conversation between a Dawah Bot ("AI") and a human ("User").
Transform each extracted Q&A into a standalone, formal FAQ entry — it should read as if it belongs in a Q&A knowledge base, not extracted from a chat.

### CONVERSATION DATA:
{conversation}

### CRITICAL RULES — READ CAREFULLY:

#### WHO TO EXTRACT FROM:
- ✅ ONLY extract questions asked BY THE USER ("User:" lines).
- ❌ NEVER extract questions asked BY THE BOT ("AI:" lines).
  The bot frequently asks rhetorical questions, reflective questions, and Socratic dialogue questions.
  ALL of these must be IGNORED. Only the USER's inquiries count.

#### WHAT COUNTS AS A QUESTION:
- ✅ Explicit questions (with question marks) from the user.
- ✅ Implicit requests for information or guidance (e.g., "Tell me about fasting" → user is seeking knowledge).
- ❌ Do NOT extract user confirmations ("Yes", "OK", "That's good", "Indeed", "I agree").
- ❌ Do NOT extract pleasantries ("Thank you", "Goodbye", "Salam alaikum" with no follow-up question).
- ❌ Do NOT extract emotional statements UNLESS they clearly seek advice (e.g., "I feel lost" alone = not a question; "I feel lost, what should I do?" = question).
- ❌ Do NOT extract user answers to the bot's demographic/profiling questions (e.g., user saying "I am Christian from India" in response to bot asking for info).
- ❌ Do NOT extract meta-questions about the bot itself (e.g., "What can you do?", "Who made you?", "Do you speak Arabic?", "Can I write in Czech?", "Which LLM are you?"). These have no Islamic Q&A value.
- ❌ Do NOT extract utility/service requests where the user asks the bot to rephrase, summarize, translate, rewrite, or edit text. These are tool-usage requests, not Islamic questions. Exception: if the user's underlying request contains a genuine Islamic inquiry (e.g., "Summarize what Islam says about fasting"), extract the Islamic question, not the rephrase request.

#### DO NOT HALLUCINATE:
- Only extract questions that are EXPLICITLY present in the user's messages.
- Do NOT infer, fabricate, or generate questions that the user never asked.
- If the user sent 2 messages, you cannot extract 5 questions. Count carefully.

### EXTRACTION FIELDS:

1. **conversation_status**: Classify the conversation as ONE of:
   * "has_qa" — The user asked at least one valid question that the bot answered (or attempted to answer).
   * "unanswered" — The user asked a real question, but the bot ONLY requested demographic info and NEVER provided a substantive answer.
   * "greeting_only" — The user only sent a greeting (salam, hello, hi) with no substantive follow-up.
   * "profiling_only" — The conversation consists only of the bot asking for demographics and the user answering (e.g., "I am from India", "I speak Arabic").
   * "no_user_response" — The bot sent a message but the user never responded at all.
   * "off_topic" — The conversation happened but was completely unrelated to Islam, religion, or the bot's purpose.
   * "minimal_engagement" — The user sent something (emoji, gibberish, single word) but no real question or topic.

2. **has_questions**: true if at least one Q&A pair was extracted, false otherwise.

3. **total_questions**: The total number of Q&A pairs extracted.

4. **qa_pairs**: For each extracted pair:
   - **question_number**: Sequential number starting from 1.
   - **question**: Rewrite the user's question into a clear, formal, standalone English question.
     * It must be self-contained and understandable WITHOUT reading the conversation.
     * Resolve all pronouns and vague references (e.g., "Why do you do that?" → "Why do Muslims fast during Ramadan?").
     * Preserve SPECIFIC DETAILS from the user's message — do not make it generic.
       ❌ BAD: "What can I do about my current situation?"
       ✅ GOOD: "What should a Muslim in a wheelchair do if they are facing housing difficulties and being offered only a hospice?"
     * Translate non-English questions to English.
   - **answer**: Write a clear, comprehensive English answer based on the bot's response.
     * Remove ALL conversational filler ("Great question!", "Welcome!", "Thank you for sharing", "Let me know if...").
     * Do NOT over-summarize. Preserve the core theological arguments, key evidence, and Quranic/Hadith references cited by the bot.
     * The answer must be FAITHFUL to what the bot actually said — do not add information the bot did not provide.
      * If the bot NEVER answered the question (only asked for demographics), do NOT extract it as a Q&A pair. Set conversation_status to "unanswered" instead.
   - **topic_category**: Choose ONE:
     * "Aqeedah" — Beliefs, theology, God's nature, monotheism, afterlife, divine attributes.
     * "Fiqh" — Islamic rulings, halal/haram, prayer, fasting, zakat, practical worship, financial transactions.
     * "Comparative Religion" — Comparing Islam with Christianity, Judaism, Atheism, etc. Including discussing Biblical verses.
     * "Quran & Tafsir" — Questions about Quranic verses, interpretation, or recitation.
     * "Hadith & Seerah" — Prophetic traditions, their authenticity, Prophet Muhammad's life, companions.
     * "Islamic History" — Historical events, caliphates, civilizations, genealogy.
     * "Personal Advice" — Personal problems, life guidance, emotional support through Islamic lens.
     * "Ethics & Morality" — Moral questions, behavior, manners in Islam.
     * "Dawah Methodology" — How to do dawah, responding to arguments, outreach strategy, crafting responses.
     * "New Muslim Support" — Questions from new converts about practicing Islam, finding community, adjusting to Islamic life.
     * "Other" — Anything that doesn't fit the above (meta-questions about the bot, language requests, etc.).
   - **is_follow_up**: true if this question builds on or refers back to a previous exchange in the same conversation. false if it's a new independent topic.

5. **Exclusions — Do NOT extract a Q&A pair if**:
    - The "question" is actually the bot's question, not the user's.
    - The user only confirmed or agreed without asking anything new.
    - The bot only responded by asking for demographic info without answering — AND the user's message was also just providing demographics (not a real question).
    - ❌ The user asked a real question BUT the bot NEVER answered it (only requested demographics). These go under "unanswered" status, NOT into qa_pairs.
    - ❌ The user's question is about the bot itself (capabilities, identity, language support, developer info). These are meta-questions with no knowledge-base value.
    - ❌ The user is asking the bot to rephrase, summarize, translate, rewrite, or proofread text. These are utility requests, not Islamic Q&A.
    - ❌ The user's question is completely unrelated to Islam, religion, faith, or the bot's dawah purpose (e.g., asking about weather, sports, coding help).

    - If no valid questions exist, return an empty qa_pairs list [].

### JSON OUTPUT (Strict JSON only, no markdown wrapping):
{{
  "conversation_status": "string",
  "has_questions": false,
  "total_questions": 0,
  "qa_pairs": [
    {{
      "question_number": 1,
      "question": "string",
      "answer": "string",
      "topic_category": "string",
      "is_follow_up": false
    }}
  ]
}}"""

print("✅ Q&A extraction prompt loaded!")



✅ Q&A extraction prompt loaded!


In [5]:

# =============================================================================
# CELL 6: LLM API Call Function (Provider-Aware) with Token Tracking
# =============================================================================

def call_llm_api(system_prompt, user_prompt, max_retries=MAX_RETRIES):
    """
    Call the configured LLM API and return response with token counts.
    Supports both OpenAI and Gemini providers.
    Includes exponential backoff for rate limiting.
    Returns: (response_text, input_tokens, output_tokens)
    """
    for attempt in range(max_retries):
        try:
            if PROVIDER == "openai":
                # Reasoning models (o3, o3-mini, o4) and GPT-5+ use different params:
                #   - max_completion_tokens instead of max_tokens
                #   - temperature is NOT supported (only default 1 is allowed)
                is_new_model = any(MODEL_ID.startswith(prefix) for prefix in ("gpt-5", "o3", "o4"))
                
                api_params = {
                    "model": MODEL_ID,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    "response_format": {"type": "json_object"},
                }
                
                if is_new_model:
                    api_params["max_completion_tokens"] = MAX_TOKENS
                    # temperature not supported — uses default (1)
                else:
                    api_params["max_tokens"] = MAX_TOKENS
                    api_params["temperature"] = TEMPERATURE
                
                response = client.chat.completions.create(**api_params)
                
                response_text = response.choices[0].message.content.strip()
                input_tokens = response.usage.prompt_tokens
                output_tokens = response.usage.completion_tokens
                
            elif PROVIDER == "gemini":
                response = client.models.generate_content(
                    model=MODEL_ID,
                    contents=user_prompt,
                    config=genai_types.GenerateContentConfig(
                        system_instruction=system_prompt,
                        temperature=TEMPERATURE,
                        max_output_tokens=MAX_TOKENS,
                        response_mime_type="application/json"
                    )
                )
                
                response_text = response.text.strip()
                # Extract token counts from Gemini usage metadata
                usage = response.usage_metadata
                input_tokens = usage.prompt_token_count
                output_tokens = usage.candidates_token_count
            
            return response_text, input_tokens, output_tokens
            
        except Exception as e:
            error_str = str(e).lower()
            
            # Check if it's a rate limit error
            if 'rate' in error_str or '429' in error_str or 'quota' in error_str or 'resource_exhausted' in error_str:
                wait_time = RETRY_BASE_DELAY * (2 ** attempt) + (attempt * 0.5)
                print(f"   ⚠️ Rate limit hit, waiting {wait_time:.1f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"   ⚠️ API error (attempt {attempt + 1}/{max_retries}): {e}")
                if attempt < max_retries - 1:
                    time.sleep(RETRY_BASE_DELAY * (2 ** attempt))
                else:
                    raise e


print("✅ LLM API function loaded!")



✅ LLM API function loaded!


In [6]:

# =============================================================================
# CELL 7: Core Processing Functions
# =============================================================================

def parse_json_response(response):
    """Parse and validate JSON response from Q&A extraction."""
    
    response = response.strip()
    
    # Remove markdown code blocks if present
    if response.startswith("```json"):
        response = response[7:]
    if response.startswith("```"):
        response = response[3:]
    if response.endswith("```"):
        response = response[:-3]
    
    response = response.strip()
    
    try:
        repaired = repair_json(response)
        result = json.loads(repaired)
        
        # Validate and set defaults for required fields
        if not isinstance(result.get("conversation_status"), str):
            result["conversation_status"] = "minimal_engagement"
        
        if not isinstance(result.get("has_questions"), bool):
            result["has_questions"] = False
        
        if not isinstance(result.get("total_questions"), int):
            result["total_questions"] = 0
        
        if not isinstance(result.get("qa_pairs"), list):
            result["qa_pairs"] = []
        
        # Validate each Q&A pair
        validated_pairs = []
        for i, pair in enumerate(result["qa_pairs"]):
            if not isinstance(pair, dict):
                continue
            validated_pair = {
                "question_number": pair.get("question_number", i + 1),
                "question": pair.get("question", ""),
                "answer": pair.get("answer", ""),
                "topic_category": pair.get("topic_category", "Other"),
                "is_follow_up": pair.get("is_follow_up", False)
            }
            validated_pairs.append(validated_pair)
        
        result["qa_pairs"] = validated_pairs
        
        # Reconcile counts
        result["total_questions"] = len(validated_pairs)
        result["has_questions"] = len(validated_pairs) > 0
        
        result["extraction_status"] = "success"
        
    except Exception as e:
        result = {
            "conversation_status": "ERROR",
            "has_questions": False,
            "total_questions": 0,
            "qa_pairs": [],
            "extraction_status": "failed",
            "error_message": str(e)
        }
    
    return result


def flatten_extraction_result(extracted):
    """
    Flatten nested JSON extraction result into a flat dictionary for DataFrame.
    """
    flat = {
        "conversation_status": extracted.get("conversation_status", ""),
        "has_questions": extracted.get("has_questions", False),
        "total_questions": extracted.get("total_questions", 0),
        "qa_pairs": json.dumps(extracted.get("qa_pairs", []), ensure_ascii=False),
        "extraction_status": extracted.get("extraction_status", "unknown")
    }
    
    return flat


def extract_from_conversation(conversation_text):
    """
    Extract Q&A pairs directly from a conversation.
    Returns: (parsed_result, input_tokens, output_tokens)
    """
    user_content = EXTRACT_USER_PROMPT_TEMPLATE.format(
        conversation=conversation_text
    )
    
    response_text, input_tokens, output_tokens = call_llm_api(
        EXTRACT_SYSTEM_PROMPT, user_content
    )
    
    parsed_result = parse_json_response(response_text)
    
    return parsed_result, input_tokens, output_tokens


def process_single_row(row_data):
    """
    Process a single conversation: extract Q&A pairs.
    Designed for parallel execution.
    
    Args:
        row_data: tuple of (index, row_dict)
    
    Returns: 
        tuple of (index, result_dict)
    """
    idx, row = row_data
    
    conversation_id = row.get('general_chat_id', 'unknown')
    conversation = str(row.get('full_conversation', ''))
    
    # Initialize result with all expected columns
    result = {
        'conversation_id': conversation_id,
        # Extracted fields
        'conversation_status': '',
        'has_questions': False,
        'total_questions': 0,
        'qa_pairs': '[]',
        # Processing metadata
        'processing_status': 'pending',
        'input_tokens': 0,
        'output_tokens': 0,
        'total_tokens': 0
    }
    
    try:
        # Extract Q&A pairs from conversation
        extracted, input_tokens, output_tokens = extract_from_conversation(conversation)
        
        # Flatten the nested result
        flat_result = flatten_extraction_result(extracted)
        
        # Update result with flattened data
        result.update(flat_result)
        result['processing_status'] = flat_result.get('extraction_status', 'success')
        
        # Token counts
        result['input_tokens'] = input_tokens
        result['output_tokens'] = output_tokens
        result['total_tokens'] = input_tokens + output_tokens
            
    except Exception as e:
        result['processing_status'] = f'error: {str(e)}'
    
    return (idx, result)


print("✅ Core functions loaded!")



✅ Core functions loaded!


In [7]:

# =============================================================================
# CELL 8: Thread-Safe Checkpointing Functions
# =============================================================================

# Thread-safe lock for checkpointing
checkpoint_lock = threading.Lock()


def load_checkpoint():
    """Load checkpoint if exists, return processed results."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
                checkpoint = json.load(f)
            print(f"✅ Checkpoint found! {len(checkpoint['completed_indices'])} rows already processed")
            return checkpoint
        except Exception as e:
            print(f"⚠️ Checkpoint file corrupted, starting fresh: {e}")
            return None
    return None


def save_checkpoint(completed_indices, results_dict):
    """
    Thread-safe checkpoint saving.
    
    Args:
        completed_indices: set of completed row indices
        results_dict: dict mapping index to result
    """
    with checkpoint_lock:
        checkpoint = {
            'completed_indices': list(completed_indices),
            'results': {str(k): v for k, v in results_dict.items()},
            'model_id': MODEL_ID,
            'timestamp': datetime.now().isoformat()
        }
        
        # Save to temp file first, then rename (atomic operation)
        temp_file = CHECKPOINT_FILE + ".tmp"
        with open(temp_file, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False)
        
        os.replace(temp_file, CHECKPOINT_FILE)


def delete_checkpoint():
    """Delete checkpoint file after successful completion."""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("✅ Checkpoint file cleaned up")


print("✅ Thread-safe checkpointing functions loaded!")



✅ Thread-safe checkpointing functions loaded!


In [8]:

# =============================================================================
# CELL 9: Main Processing (Mode-Aware)
# =============================================================================

# Load data (shared by both modes)
print("="*60)
print(f"STARTING Q&A EXTRACTION WITH {MODEL_ID.upper()}")
print(f"Provider: {PROVIDER.upper()}")
print(f"Mode: {PROCESSING_MODE.upper()}")
print("="*60)

print(f"\nLoading data from: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE)
total_rows = len(df)
print(f"Total conversations: {total_rows}")

start_time = time.time()


# =============================================================================
# CELL 9A: PARALLEL MODE (existing behavior)
# =============================================================================

if PROCESSING_MODE == "parallel":
    print(f"\n⚡ PARALLEL MODE — {NUM_WORKERS} workers")
    print("-"*60)

    # Check for checkpoint
    checkpoint = load_checkpoint()

    if checkpoint:
        # Verify checkpoint is for the same model
        if checkpoint.get('model_id') != MODEL_ID:
            print(f"⚠️ Checkpoint is for different model ({checkpoint.get('model_id')})")
            print(f"   Current model: {MODEL_ID}")
            user_input = input("   Start fresh? (y/n): ")
            if user_input.lower() == 'y':
                completed_indices = set()
                results_dict = {}
            else:
                completed_indices = set(checkpoint['completed_indices'])
                results_dict = {int(k): v for k, v in checkpoint['results'].items()}
        else:
            completed_indices = set(checkpoint['completed_indices'])
            results_dict = {int(k): v for k, v in checkpoint['results'].items()}
            print(f"   Resuming... {len(completed_indices)} already done, {total_rows - len(completed_indices)} remaining")
    else:
        completed_indices = set()
        results_dict = {}

    # Prepare rows to process (skip already completed)
    rows_to_process = []
    for idx in range(total_rows):
        if idx not in completed_indices:
            row = df.iloc[idx].to_dict()
            rows_to_process.append((idx, row))

    print(f"\nRows to process: {len(rows_to_process)}")
    print("-"*60)

    # Tracking variables
    processed_count = len(completed_indices)
    last_checkpoint_count = processed_count

    # Progress bar
    pbar = tqdm(total=total_rows, initial=processed_count, desc="Extracting Q&A")

    # Thread-safe counter
    counter_lock = threading.Lock()


    def update_progress(idx, result):
        """Thread-safe progress update."""
        global processed_count, last_checkpoint_count
        
        with counter_lock:
            results_dict[idx] = result
            completed_indices.add(idx)
            processed_count += 1
            pbar.update(1)
            
            # Show brief status
            conv_id = result['conversation_id']
            status = result['processing_status']
            q_count = result['total_questions']
            pbar.set_postfix({
                'ID': str(conv_id)[:10],
                'Qs': q_count,
                'Status': 'OK' if status == 'success' else 'ERR'
            })
            
            # Save checkpoint periodically
            if processed_count - last_checkpoint_count >= CHECKPOINT_EVERY:
                save_checkpoint(completed_indices, results_dict)
                last_checkpoint_count = processed_count


    # Process in parallel
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
            # Submit all tasks
            future_to_idx = {
                executor.submit(process_single_row, row_data): row_data[0] 
                for row_data in rows_to_process
            }
            
            # Process completed tasks
            for future in concurrent.futures.as_completed(future_to_idx):
                try:
                    idx, result = future.result()
                    update_progress(idx, result)
                except Exception as e:
                    idx = future_to_idx[future]
                    error_result = {
                        'conversation_id': df.iloc[idx].to_dict().get('general_chat_id', 'unknown'),
                        'conversation_status': '',
                        'has_questions': False,
                        'total_questions': 0,
                        'qa_pairs': '[]',
                        'processing_status': f'error: {str(e)}',
                        'input_tokens': 0,
                        'output_tokens': 0,
                        'total_tokens': 0
                    }
                    update_progress(idx, error_result)

    except KeyboardInterrupt:
        print("\n\n⚠️ Processing interrupted by user!")
        print("Saving checkpoint before exit...")
        save_checkpoint(completed_indices, results_dict)
        pbar.close()
        print(f"Progress saved: {len(completed_indices)}/{total_rows} completed")
        print("You can resume later by running this cell again.")
        raise

    except Exception as e:
        print(f"\n\n❌ Error occurred: {e}")
        print("Saving checkpoint before exit...")
        save_checkpoint(completed_indices, results_dict)
        pbar.close()
        print("You can resume later by running this cell again.")
        raise

    # Final save
    pbar.close()
    save_checkpoint(completed_indices, results_dict)

    # Calculate timing
    elapsed_time = time.time() - start_time
    rows_processed_this_session = len(rows_to_process)
    rate = rows_processed_this_session / elapsed_time if elapsed_time > 0 else 0

    print("\n" + "="*60)
    print("✅ Q&A EXTRACTION COMPLETE!")
    print("="*60)
    print(f"Total processed: {len(completed_indices)} rows")
    print(f"This session: {rows_processed_this_session} rows in {elapsed_time:.1f}s")
    print(f"Processing rate: {rate:.2f} conversations/second")


# =============================================================================
# CELL 9B: BATCH MODE (Gemini Batch API)
# =============================================================================

elif PROCESSING_MODE == "batch":
    print(f"\n📦 BATCH MODE — Gemini Batch API (50% cost savings)")
    print("-"*60)

    results_dict = {}

    # ------------------------------------------------------------------
    # Check if a batch job was already submitted (resume support)
    # ------------------------------------------------------------------
    existing_job_name = None
    if os.path.exists(BATCH_JOB_FILE):
        try:
            with open(BATCH_JOB_FILE, 'r') as f:
                job_info = json.load(f)
            existing_job_name = job_info.get('job_name')
            print(f"📋 Found existing batch job: {existing_job_name}")
        except Exception as e:
            print(f"⚠️ Could not read batch job file: {e}")
            existing_job_name = None

    # ------------------------------------------------------------------
    # Step 1: Prepare JSONL & Submit (skip if job already exists)
    # ------------------------------------------------------------------
    if existing_job_name is None:
        print("\n📝 Step 1: Preparing batch requests JSONL file...")

        # Build the JSONL file
        request_count = 0
        with open(BATCH_JSONL_FILE, 'w', encoding='utf-8') as f:
            for idx in range(total_rows):
                row = df.iloc[idx]
                conversation_text = str(row.get('full_conversation', ''))
                user_content = EXTRACT_USER_PROMPT_TEMPLATE.format(
                    conversation=conversation_text
                )

                # Each line is a JSON object with key + GenerateContentRequest
                # NOTE: Field names use camelCase (raw REST API format) for Batch API compatibility
                batch_line = {
                    "key": f"row-{idx}",
                    "request": {
                        "contents": [
                            {
                                "parts": [{"text": user_content}],
                                "role": "user"
                            }
                        ],
                        "systemInstruction": {
                            "parts": [{"text": EXTRACT_SYSTEM_PROMPT}]
                        },
                        "generationConfig": {
                            "temperature": TEMPERATURE,
                            "maxOutputTokens": MAX_TOKENS,
                            "responseMimeType": "application/json"
                        }
                    }
                }
                f.write(json.dumps(batch_line, ensure_ascii=False) + "\n")
                request_count += 1

        file_size_mb = os.path.getsize(BATCH_JSONL_FILE) / (1024 * 1024)
        print(f"   ✅ Created {BATCH_JSONL_FILE}: {request_count} requests, {file_size_mb:.2f} MB")

        # Upload JSONL via File API
        print("\n📤 Step 2: Uploading JSONL file to Gemini File API...")
        uploaded_file = client.files.upload(
            file=BATCH_JSONL_FILE,
            config=genai_types.UploadFileConfig(
                display_name=f"qa-batch-{MODEL_ID}",
                mime_type="jsonl"
            )
        )
        print(f"   ✅ Uploaded: {uploaded_file.name}")

        # Submit batch job
        print(f"\n🚀 Step 3: Submitting batch job with model '{MODEL_ID}'...")
        batch_job = client.batches.create(
            model=MODEL_ID,
            src=uploaded_file.name,
            config={
                'display_name': f"qa-extraction-{MODEL_ID}-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
            },
        )
        job_name = batch_job.name
        print(f"   ✅ Batch job created: {job_name}")

        # Save job name for resume
        with open(BATCH_JOB_FILE, 'w') as f:
            json.dump({
                'job_name': job_name,
                'model_id': MODEL_ID,
                'total_rows': total_rows,
                'submitted_at': datetime.now().isoformat()
            }, f, indent=2)
        print(f"   💾 Job info saved to: {BATCH_JOB_FILE}")

    else:
        job_name = existing_job_name
        print(f"   ⏩ Skipping submission — using existing job: {job_name}")

    # ------------------------------------------------------------------
    # Step 4: Poll for completion
    # ------------------------------------------------------------------
    print(f"\n⏳ Step 4: Polling batch job status (every {BATCH_POLL_INTERVAL}s)...")
    print("   Press Ctrl+C to stop polling. You can resume later.")

    completed_states = {
        'JOB_STATE_SUCCEEDED',
        'JOB_STATE_FAILED',
        'JOB_STATE_CANCELLED',
        'JOB_STATE_EXPIRED',
    }

    # Use clear_output in notebooks to prevent output bloat
    try:
        from IPython.display import clear_output
        _is_notebook = True
    except ImportError:
        _is_notebook = False

    try:
        poll_start = time.time()
        poll_count = 0
        while True:
            batch_job = client.batches.get(name=job_name)
            state = batch_job.state.name if hasattr(batch_job.state, 'name') else str(batch_job.state)
            elapsed = time.time() - poll_start
            poll_count += 1

            # In notebooks: clear and rewrite to prevent output bloat
            if _is_notebook and poll_count % 5 == 0:
                clear_output(wait=True)
                print(f"⏳ Polling batch job: {job_name}")
                print(f"   Polls so far: {poll_count}")

            # Always show current status (overwrites in terminal, accumulates minimally in notebook)
            print(f"   [{datetime.now().strftime('%H:%M:%S')}] State: {state} | Elapsed: {elapsed/60:.1f} min")

            if state in completed_states:
                break

            time.sleep(BATCH_POLL_INTERVAL)

        # Final clear summary
        if _is_notebook:
            clear_output(wait=True)
        print(f"\n⏳ Polling complete after {poll_count} polls ({elapsed/60:.1f} min)")
        print(f"   Final state: {state}")

    except KeyboardInterrupt:
        print(f"\n\n⚠️ Polling interrupted! Job '{job_name}' is still running.")
        print(f"   Run this cell again to resume polling.")
        print(f"   Job info saved in: {BATCH_JOB_FILE}")
        raise

    # ------------------------------------------------------------------
    # Step 5: Download & Parse Results
    # ------------------------------------------------------------------
    final_state = batch_job.state.name if hasattr(batch_job.state, 'name') else str(batch_job.state)

    if final_state == 'JOB_STATE_SUCCEEDED':
        print(f"\n✅ Batch job SUCCEEDED!")
        print("📥 Step 5: Downloading and parsing results...")

        # Save results to disk first (safer for 12K+ conversations — avoids OOM)
        BATCH_RAW_RESULTS_FILE = BATCH_JSONL_FILE.replace('.jsonl', '_results.jsonl')

        # Determine result source: file or inline
        if hasattr(batch_job, 'dest') and batch_job.dest and hasattr(batch_job.dest, 'file_name') and batch_job.dest.file_name:
            # Results are in a file — download to disk
            result_file_name = batch_job.dest.file_name
            print(f"   Downloading result file: {result_file_name}")
            file_content = client.files.download(file=result_file_name)
            with open(BATCH_RAW_RESULTS_FILE, 'wb') as f:
                f.write(file_content)
            print(f"   💾 Saved raw results to: {BATCH_RAW_RESULTS_FILE}")

        elif hasattr(batch_job, 'dest') and batch_job.dest and hasattr(batch_job.dest, 'inlined_responses') and batch_job.dest.inlined_responses:
            # Results are inline — save to disk first
            print(f"   Processing inline responses...")
            with open(BATCH_RAW_RESULTS_FILE, 'w', encoding='utf-8') as f:
                for resp in batch_job.dest.inlined_responses:
                    if resp.response:
                        f.write(json.dumps({
                            "key": getattr(resp, 'key', ''),
                            "response": {"text": resp.response.text}
                        }, ensure_ascii=False) + '\n')
            print(f"   💾 Saved inline results to: {BATCH_RAW_RESULTS_FILE}")
        else:
            print("❌ No results found in batch job response!")
            BATCH_RAW_RESULTS_FILE = None

        # Read results from disk for parsing
        if BATCH_RAW_RESULTS_FILE and os.path.exists(BATCH_RAW_RESULTS_FILE):
            with open(BATCH_RAW_RESULTS_FILE, 'r', encoding='utf-8') as f:
                result_lines = [line.strip() for line in f if line.strip()]
        else:
            result_lines = []

        # Parse each result line and map back to row indices
        print(f"   Parsing {len(result_lines)} result lines...")
        parsed_count = 0
        error_count = 0

        for line in result_lines:
            try:
                result_obj = json.loads(line)
                key = result_obj.get("key", "")

                # Extract row index from key "row-{idx}"
                if key.startswith("row-"):
                    idx = int(key.split("-", 1)[1])
                else:
                    continue

                # Get the response text
                response_data = result_obj.get("response", {})
                if isinstance(response_data, dict):
                    # File-based results: response contains candidates
                    candidates = response_data.get("candidates", [])
                    if candidates:
                        parts = candidates[0].get("content", {}).get("parts", [])
                        response_text = parts[0].get("text", "") if parts else ""
                    else:
                        response_text = response_data.get("text", "")
                else:
                    response_text = str(response_data)

                # Parse & validate JSON from LLM response
                parsed_result = parse_json_response(response_text)
                flat_result = flatten_extraction_result(parsed_result)

                # Extract token counts from usage metadata if available
                usage = response_data.get("usageMetadata", {})
                input_tokens = usage.get("promptTokenCount", 0)
                output_tokens = usage.get("candidatesTokenCount", 0)

                conversation_id = df.iloc[idx].get('general_chat_id', 'unknown')

                results_dict[idx] = {
                    'conversation_id': conversation_id,
                    **flat_result,
                    'processing_status': flat_result.get('extraction_status', 'success'),
                    'input_tokens': input_tokens,
                    'output_tokens': output_tokens,
                    'total_tokens': input_tokens + output_tokens,
                }
                parsed_count += 1

            except Exception as e:
                error_count += 1
                # Try to extract idx from the key if possible
                try:
                    idx = int(key.split("-", 1)[1])
                    conversation_id = df.iloc[idx].get('general_chat_id', 'unknown')
                except:
                    idx = -1
                    conversation_id = 'unknown'

                if idx >= 0:
                    results_dict[idx] = {
                        'conversation_id': conversation_id,
                        'conversation_status': 'ERROR',
                        'has_questions': False,
                        'total_questions': 0,
                        'qa_pairs': '[]',
                        'processing_status': f'batch_parse_error: {str(e)}',
                        'input_tokens': 0,
                        'output_tokens': 0,
                        'total_tokens': 0,
                    }

        # Fill in any missing rows (requests that got no response)
        for idx in range(total_rows):
            if idx not in results_dict:
                results_dict[idx] = {
                    'conversation_id': df.iloc[idx].get('general_chat_id', 'unknown'),
                    'conversation_status': 'ERROR',
                    'has_questions': False,
                    'total_questions': 0,
                    'qa_pairs': '[]',
                    'processing_status': 'batch_no_response',
                    'input_tokens': 0,
                    'output_tokens': 0,
                    'total_tokens': 0,
                }

        elapsed_time = time.time() - start_time

        print(f"\n   ✅ Parsed: {parsed_count} | Errors: {error_count} | Missing: {total_rows - parsed_count - error_count}")
        print(f"   Total time (including wait): {elapsed_time/60:.1f} minutes")

        # Clean up batch job file
        if os.path.exists(BATCH_JOB_FILE):
            os.remove(BATCH_JOB_FILE)
            print(f"   🗑️ Cleaned up {BATCH_JOB_FILE}")

    else:
        # Job failed / cancelled / expired
        print(f"\n❌ Batch job ended with state: {final_state}")
        if hasattr(batch_job, 'error') and batch_job.error:
            print(f"   Error: {batch_job.error}")
        print(f"   You may need to resubmit. Delete '{BATCH_JOB_FILE}' to start fresh.")

        # Fill results_dict with error entries so downstream cells don't crash
        for idx in range(total_rows):
            results_dict[idx] = {
                'conversation_id': df.iloc[idx].get('general_chat_id', 'unknown'),
                'conversation_status': 'ERROR',
                'has_questions': False,
                'total_questions': 0,
                'qa_pairs': '[]',
                'processing_status': f'batch_{final_state.lower()}',
                'input_tokens': 0,
                'output_tokens': 0,
                'total_tokens': 0,
            }

        elapsed_time = time.time() - start_time

    print("\n" + "="*60)
    print("✅ BATCH PROCESSING COMPLETE!")
    print("="*60)




⏳ Polling complete after 1 polls (0.0 min)
   Final state: JOB_STATE_SUCCEEDED

✅ Batch job SUCCEEDED!
📥 Step 5: Downloading and parsing results...
   💾 Saved raw results to: batch_requests_results.jsonl
   Parsing 12448 result lines...

   ✅ Parsed: 12448 | Errors: 0 | Missing: 0
   Total time (including wait): 0.2 minutes
   🗑️ Cleaned up batch_job_gemini_3_1_flash_lite_preview.json

✅ BATCH PROCESSING COMPLETE!


In [9]:

# =============================================================================
# CELL 10: Merge Results and Save to CSV
# =============================================================================

print("="*60)
print("MERGING RESULTS")
print("="*60)

# Convert results_dict to ordered list matching DataFrame
results = [results_dict[idx] for idx in range(total_rows)]

# Merge with original dataframe
print("\nMerging results with original data...")

# Define new columns to add
new_columns = [
    'conversation_status',
    'has_questions',
    'total_questions',
    'qa_pairs',
    'processing_status',
    'input_tokens',
    'output_tokens',
    'total_tokens'
]

# Add new columns to original dataframe
for col in new_columns:
    df[col] = [results_dict.get(idx, {}).get(col, '') for idx in range(total_rows)]

# Convert total_questions to numeric
df['total_questions'] = pd.to_numeric(df['total_questions'], errors='coerce').fillna(0).astype(int)

# Convert has_questions to boolean
df['has_questions'] = df['has_questions'].apply(lambda x: x if isinstance(x, bool) else str(x).lower() == 'true')

# Save final output
print(f"\nSaving to: {OUTPUT_FILE}")
df.to_csv(OUTPUT_FILE, index=False)
print("✅ CSV file saved successfully!")

# Clean up checkpoint
delete_checkpoint()



MERGING RESULTS

Merging results with original data...

Saving to: Islam_chat_questions_extraction.csv
✅ CSV file saved successfully!


In [10]:

# =============================================================================
# CELL 11: Save Results to JSON File
# =============================================================================

print("\n" + "="*60)
print("SAVING RESULTS TO JSON")
print("="*60)

# Define JSON output filename
JSON_OUTPUT_FILE = f"Islam_chat_questions_extraction.json"

# Prepare results for JSON export
json_results = []

for idx, row in df.iterrows():
    # Parse qa_pairs back from string
    qa_pairs_raw = row.get('qa_pairs', '[]')
    if isinstance(qa_pairs_raw, str):
        try:
            qa_pairs = json.loads(qa_pairs_raw)
        except:
            qa_pairs = []
    else:
        qa_pairs = qa_pairs_raw if isinstance(qa_pairs_raw, list) else []
    
    conv_result = {
        "conversation_id": row.get('general_chat_id', ''),
        "conversation_status": row.get('conversation_status', ''),
        "has_questions": bool(row.get('has_questions', False)),
        "total_questions": int(row.get('total_questions', 0)),
        "qa_pairs": qa_pairs,
        "processing_metadata": {
            "status": row.get('processing_status', ''),
            "input_tokens": int(row.get('input_tokens', 0)),
            "output_tokens": int(row.get('output_tokens', 0)),
            "total_tokens": int(row.get('total_tokens', 0))
        }
    }
    
    json_results.append(conv_result)

# Create final JSON structure
json_output = {
    "metadata": {
        "provider": PROVIDER,
        "model_id": MODEL_ID,
        "total_conversations": len(json_results),
        "conversations_with_qa": len(df[df['has_questions'] == True]),
        "conversations_without_qa": len(df[df['has_questions'] == False]),
        "total_qa_pairs_extracted": int(df['total_questions'].sum()),
        "generated_at": datetime.now().isoformat()
    },
    "results": json_results
}

# Save to JSON file
with open(JSON_OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(json_output, f, indent=2, ensure_ascii=False)

print(f"✅ JSON file saved to: {JSON_OUTPUT_FILE}")
print(f"   Total conversations: {len(json_results)}")
print(f"   Total Q&A pairs: {int(df['total_questions'].sum())}")
print(f"   File size: {os.path.getsize(JSON_OUTPUT_FILE) / 1024:.2f} KB")




SAVING RESULTS TO JSON
✅ JSON file saved to: Islam_chat_questions_extraction.json
   Total conversations: 12448
   Total Q&A pairs: 14231
   File size: 15941.89 KB


In [11]:

# =============================================================================
# CELL 12: Token Statistics
# =============================================================================

print("\n" + "="*60)
print(f"TOKEN STATISTICS FOR {PROVIDER.upper()} / {MODEL_ID.upper()}")
print("="*60)

# Convert token columns to numeric
token_columns = ['input_tokens', 'output_tokens', 'total_tokens']

for col in token_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Filter successful rows only
successful_df = df[df['processing_status'] == 'success']

print(f"\nSuccessful conversations: {len(successful_df)} / {len(df)}")

# Calculate statistics
def calc_stats(series):
    """Calculate mean and median for a series."""
    values = series[series > 0].tolist()
    if not values:
        return {'mean': 0, 'median': 0, 'min': 0, 'max': 0, 'total': 0}
    return {
        'mean': round(statistics.mean(values), 2),
        'median': round(statistics.median(values), 2),
        'min': min(values),
        'max': max(values),
        'total': sum(values)
    }

# Token statistics
print("\n📊 TOKEN USAGE:")
input_stats = calc_stats(successful_df['input_tokens'])
output_stats = calc_stats(successful_df['output_tokens'])
total_stats = calc_stats(successful_df['total_tokens'])

print(f"   Input tokens  - Mean: {input_stats['mean']:,.0f} | Median: {input_stats['median']:,.0f} | Total: {input_stats['total']:,}")
print(f"   Output tokens - Mean: {output_stats['mean']:,.0f} | Median: {output_stats['median']:,.0f} | Total: {output_stats['total']:,}")
print(f"   Total tokens  - Mean: {total_stats['mean']:,.0f} | Median: {total_stats['median']:,.0f} | Total: {total_stats['total']:,}")

# Grand totals
grand_total_input = input_stats['total']
grand_total_output = output_stats['total']
print(f"\n🔢 GRAND TOTALS:")
print(f"   Total input tokens:  {grand_total_input:,}")
print(f"   Total output tokens: {grand_total_output:,}")
print(f"   Total all tokens:    {grand_total_input + grand_total_output:,}")

# Save statistics to JSON
stats_report = {
    "provider": PROVIDER,
    "model_id": MODEL_ID,
    "num_workers": NUM_WORKERS,
    "total_conversations": len(df),
    "successful_conversations": len(successful_df),
    "tokens": {
        "input": input_stats,
        "output": output_stats,
        "total": total_stats
    },
    "grand_totals": {
        "input_tokens": grand_total_input,
        "output_tokens": grand_total_output,
        "all_tokens": grand_total_input + grand_total_output
    },
    "timestamp": datetime.now().isoformat()
}

stats_file = f"qa_token_stats_{MODEL_ID.replace('.', '_').replace('-', '_')}.json"
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats_report, f, indent=2)
print(f"\n✅ Token statistics saved to: {stats_file}")




TOKEN STATISTICS FOR GEMINI / GEMINI-3.1-FLASH-LITE-PREVIEW

Successful conversations: 12438 / 12448

📊 TOKEN USAGE:
   Input tokens  - Mean: 3,572 | Median: 2,402 | Total: 44,428,566
   Output tokens - Mean: 270 | Median: 207 | Total: 3,352,119
   Total tokens  - Mean: 3,842 | Median: 2,608 | Total: 47,780,685

🔢 GRAND TOTALS:
   Total input tokens:  44,428,566
   Total output tokens: 3,352,119
   Total all tokens:    47,780,685

✅ Token statistics saved to: qa_token_stats_gemini_3_1_flash_lite_preview.json


In [12]:

# =============================================================================
# CELL 13: Cost Estimation (Dynamic - Provider & Model Aware)
# =============================================================================

print("\n" + "="*60)
print(f"COST ESTIMATION ({PROVIDER.upper()} / {MODEL_ID})")
print("="*60)

# Lookup pricing dynamically
if MODEL_ID in MODEL_PRICING:
    pricing = MODEL_PRICING[MODEL_ID]
    PRICE_INPUT = pricing["input"]
    PRICE_OUTPUT = pricing["output"]
else:
    print(f"⚠️  Model '{MODEL_ID}' not in pricing dictionary. Showing $0 costs.")
    PRICE_INPUT = 0
    PRICE_OUTPUT = 0

# Apply 50% batch discount
if PROCESSING_MODE == "batch":
    PRICE_INPUT *= 0.5
    PRICE_OUTPUT *= 0.5
    print(f"\n🏷️ BATCH MODE DISCOUNT: 50% off applied!")

# Calculate costs
input_cost = (grand_total_input / 1_000_000) * PRICE_INPUT
output_cost = (grand_total_output / 1_000_000) * PRICE_OUTPUT
total_cost = input_cost + output_cost

# Per conversation cost (using mean)
per_conv_input = (input_stats['mean'] / 1_000_000) * PRICE_INPUT
per_conv_output = (output_stats['mean'] / 1_000_000) * PRICE_OUTPUT
per_conv_total = per_conv_input + per_conv_output

print(f"\n💰 Pricing for {MODEL_ID}:")
print(f"   Input:  ${PRICE_INPUT:.2f} / 1M tokens")
print(f"   Output: ${PRICE_OUTPUT:.2f} / 1M tokens")

print(f"\n💵 TOTAL COST (this run):")
print(f"   Input cost:  ${input_cost:.4f}")
print(f"   Output cost: ${output_cost:.4f}")
print(f"   Total cost:  ${total_cost:.4f}")

print(f"\n💵 ESTIMATED COST PER CONVERSATION (using mean):")
print(f"   Per conversation: ${per_conv_total:.6f}")
print(f"   Per 1,000 conversations: ${per_conv_total * 1000:.4f}")
print(f"   Per 10,000 conversations: ${per_conv_total * 10000:.2f}")

# Add cost to stats report
stats_report["cost_estimation"] = {
    "pricing_per_1m_tokens": {"input": PRICE_INPUT, "output": PRICE_OUTPUT},
    "total_cost": {
        "input": round(input_cost, 6),
        "output": round(output_cost, 6),
        "total": round(total_cost, 6)
    },
    "per_conversation_cost": {
        "mean": round(per_conv_total, 8),
        "per_1000": round(per_conv_total * 1000, 4),
        "per_10000": round(per_conv_total * 10000, 2)
    }
}

# Update stats file with cost info
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats_report, f, indent=2)




COST ESTIMATION (GEMINI / gemini-3.1-flash-lite-preview)

🏷️ BATCH MODE DISCOUNT: 50% off applied!

💰 Pricing for gemini-3.1-flash-lite-preview:
   Input:  $0.12 / 1M tokens
   Output: $0.75 / 1M tokens

💵 TOTAL COST (this run):
   Input cost:  $5.5536
   Output cost: $2.5141
   Total cost:  $8.0677

💵 ESTIMATED COST PER CONVERSATION (using mean):
   Per conversation: $0.000649
   Per 1,000 conversations: $0.6486
   Per 10,000 conversations: $6.49


In [13]:

# =============================================================================
# CELL 14: Final Summary & Distributions
# =============================================================================

print("\n" + "="*60)
print("FINAL SUMMARY & DISTRIBUTIONS")
print("="*60)

print(f"\n📊 Provider: {PROVIDER.upper()}")
print(f"📊 Model: {MODEL_ID}")
print(f"📊 Workers: {NUM_WORKERS}")
print(f"📊 Total conversations processed: {len(df)}")

# Processing status breakdown
print("\n📈 Processing Status:")
status_counts = df['processing_status'].value_counts()
for status, count in status_counts.items():
    pct = (count / len(df)) * 100
    print(f"   {status}: {count} ({pct:.1f}%)")

# Conversation Status distribution
print("\n📂 Conversation Status Distribution:")
conv_status_counts = df['conversation_status'].value_counts()
for cstatus, count in conv_status_counts.items():
    pct = (count / len(df)) * 100
    print(f"   {cstatus}: {count} ({pct:.1f}%)")

# Questions per conversation
print("\n❓ Questions Per Conversation:")
q_counts = df['total_questions'].value_counts().sort_index()
for q_count, count in q_counts.items():
    pct = (count / len(df)) * 100
    bar = "█" * int(pct / 5)
    print(f"   {q_count} questions: {count} ({pct:.1f}%) {bar}")

# Has Questions breakdown
print("\n✅ Conversations With Questions:")
has_q_counts = df['has_questions'].value_counts()
for val, count in has_q_counts.items():
    pct = (count / len(df)) * 100
    label = "Has Q&A" if val else "No Q&A"
    print(f"   {label}: {count} ({pct:.1f}%)")

# Topic Category distribution (across all Q&A pairs)
print("\n📚 Topic Category Distribution (across all Q&A pairs):")
all_topics = []
for idx in range(len(df)):
    qa_raw = df.iloc[idx].get('qa_pairs', '[]')
    if isinstance(qa_raw, str):
        try:
            pairs = json.loads(qa_raw)
        except:
            pairs = []
    else:
        pairs = qa_raw if isinstance(qa_raw, list) else []
    
    for pair in pairs:
        if isinstance(pair, dict):
            all_topics.append(pair.get('topic_category', 'Other'))

if all_topics:
    topic_series = pd.Series(all_topics)
    topic_counts = topic_series.value_counts()
    for topic, count in topic_counts.items():
        pct = (count / len(all_topics)) * 100
        print(f"   {topic}: {count} ({pct:.1f}%)")
else:
    print("   No Q&A pairs extracted.")

# Follow-up questions stats
print("\n🔗 Follow-up Questions:")
all_followups = []
for idx in range(len(df)):
    qa_raw = df.iloc[idx].get('qa_pairs', '[]')
    if isinstance(qa_raw, str):
        try:
            pairs = json.loads(qa_raw)
        except:
            pairs = []
    else:
        pairs = qa_raw if isinstance(qa_raw, list) else []
    
    for pair in pairs:
        if isinstance(pair, dict):
            all_followups.append(pair.get('is_follow_up', False))

if all_followups:
    followup_count = sum(1 for f in all_followups if f)
    independent_count = len(all_followups) - followup_count
    print(f"   Independent questions: {independent_count}")
    print(f"   Follow-up questions: {followup_count}")
else:
    print("   No Q&A pairs extracted.")

# Preview of results
print("\n" + "-"*60)
print("📋 Preview of results:")
preview_cols = [
    "general_chat_id", 
    "conversation_status",
    "has_questions",
    "total_questions"
]
available_cols = [c for c in preview_cols if c in df.columns]
print(df[available_cols].head(10).to_string())

# Show sample Q&A pairs
print("\n📋 Sample Q&A pairs (first conversation with questions):")
for idx in range(len(df)):
    qa_raw = df.iloc[idx].get('qa_pairs', '[]')
    if isinstance(qa_raw, str):
        try:
            pairs = json.loads(qa_raw)
        except:
            pairs = []
    else:
        pairs = qa_raw if isinstance(qa_raw, list) else []
    
    if pairs:
        conv_id = df.iloc[idx].get('general_chat_id', 'unknown')
        print(f"\n   Conversation ID: {conv_id}")
        for pair in pairs[:3]:  # Show first 3 Q&A pairs
            print(f"   Q{pair.get('question_number', '?')}: {pair.get('question', 'N/A')}")
            print(f"   A: {pair.get('answer', 'N/A')[:150]}...")
            print(f"   Topic: {pair.get('topic_category', 'N/A')} | Follow-up: {pair.get('is_follow_up', False)}")
            print()
        break




FINAL SUMMARY & DISTRIBUTIONS

📊 Provider: GEMINI
📊 Model: gemini-3.1-flash-lite-preview
📊 Workers: 16
📊 Total conversations processed: 12448

📈 Processing Status:
   success: 12438 (99.9%)
   failed: 10 (0.1%)

📂 Conversation Status Distribution:
   has_qa: 6729 (54.1%)
   unanswered: 1962 (15.8%)
   profiling_only: 1616 (13.0%)
   greeting_only: 768 (6.2%)
   off_topic: 573 (4.6%)
   minimal_engagement: 554 (4.5%)
   other: 197 (1.6%)
   no_user_response: 39 (0.3%)
   ERROR: 10 (0.1%)

❓ Questions Per Conversation:
   0 questions: 5739 (46.1%) █████████
   1 questions: 3711 (29.8%) █████
   2 questions: 1322 (10.6%) ██
   3 questions: 670 (5.4%) █
   4 questions: 390 (3.1%) 
   5 questions: 196 (1.6%) 
   6 questions: 184 (1.5%) 
   7 questions: 79 (0.6%) 
   8 questions: 43 (0.3%) 
   9 questions: 8 (0.1%) 
   10 questions: 14 (0.1%) 
   11 questions: 48 (0.4%) 
   12 questions: 30 (0.2%) 
   14 questions: 3 (0.0%) 
   15 questions: 6 (0.0%) 
   16 questions: 1 (0.0%) 
   17 questi

In [14]:

# =============================================================================
# CELL 15: Performance Summary & Output Files
# =============================================================================

print("\n" + "="*60)
print("🏁 FINAL PERFORMANCE SUMMARY")
print("="*60)

print(f"\n⚡ PROCESSING STATS:")
print(f"   Provider: {PROVIDER.upper()}")
print(f"   Model: {MODEL_ID}")
print(f"   Mode: {PROCESSING_MODE.upper()}")
if PROCESSING_MODE == "parallel":
    print(f"   Workers used: {NUM_WORKERS}")
print(f"   Total time: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
if PROCESSING_MODE == "parallel":
    print(f"   Processing rate: {rate:.2f} conversations/second")

print(f"\n📁 OUTPUT FILES:")
print(f"   ├── {OUTPUT_FILE} (CSV)")
print(f"   ├── {JSON_OUTPUT_FILE} (JSON)")
print(f"   └── {stats_file} (Token Stats)")

print(f"\n📊 DATA SUMMARY:")
print(f"   Total conversations: {len(df)}")
success_count = len(df[df['processing_status'] == 'success'])
fail_count = len(df) - success_count
print(f"   Successful: {success_count} ({success_count/len(df)*100:.1f}%)")
print(f"   Failed: {fail_count} ({fail_count/len(df)*100:.1f}%)")
total_qa = int(df['total_questions'].sum())
print(f"   Total Q&A pairs extracted: {total_qa}")

print(f"\n💰 COST SUMMARY:")
print(f"   Total cost: ${total_cost:.4f}")
print(f"   Cost per conversation: ${per_conv_total:.6f}")

print("\n" + "="*60)
print("✅ ALL DONE!")
print("="*60)



🏁 FINAL PERFORMANCE SUMMARY

⚡ PROCESSING STATS:
   Provider: GEMINI
   Model: gemini-3.1-flash-lite-preview
   Mode: BATCH
   Total time: 13.0 seconds (0.2 minutes)

📁 OUTPUT FILES:
   ├── Islam_chat_questions_extraction.csv (CSV)
   ├── Islam_chat_questions_extraction.json (JSON)
   └── qa_token_stats_gemini_3_1_flash_lite_preview.json (Token Stats)

📊 DATA SUMMARY:
   Total conversations: 12448
   Successful: 12438 (99.9%)
   Failed: 10 (0.1%)
   Total Q&A pairs extracted: 14231

💰 COST SUMMARY:
   Total cost: $8.0677
   Cost per conversation: $0.000649

✅ ALL DONE!


In [1]:
import pandas as pd

df = pd.read_csv("S:\Midade work\Islam chat conversation analysis\Conversation question extraction\Islam_chat_questions_extraction.csv")

In [2]:
df.head()

,general_chat_id,total_messages,user_msg_count,bot_msg_count,start_time,end_time,conversation_duration,avg_response_time_user_sec,avg_response_time_ai_sec,max_gap_sec,...,conversation_language,language_confidence_score,conversation_status,has_questions,total_questions,qa_pairs,processing_status,input_tokens,output_tokens,total_tokens
0,1,28,17,11,2024-10-17 20:12:08,2025-10-29 14:53:26,376d 18h 41m 18s,957744.00,2088585.18,22974422.0,...,Arabic,0.8548,has_qa,True,2,"[{""question_number"": 1, ""question"": ""What is t...",success,2920,471,3391
1,2,7,4,3,2024-10-17 21:14:35,2024-10-18 18:17:46,0d 21h 3m 11s,25263.33,0.33,56908.0,...,Arabic,0.2713,minimal_engagement,False,0,[],success,2056,42,2098
2,3,2,1,1,2024-10-17 22:56:23,2024-10-17 22:56:24,0d 0h 0m 1s,NaN,1.00,1.0,...,English,0.9994,minimal_engagement,False,0,[],success,1962,42,2004
3,6,1,0,1,2024-10-17 19:43:10,2024-10-17 19:43:10,0d 0h 0m 0s,NaN,NaN,0.0,...,English,0.9996,minimal_engagement,False,0,[],success,1963,42,2005
4,7,1,0,1,2024-10-17 19:44:00,2024-10-17 19:44:00,0d 0h 0m 0s,NaN,NaN,0.0,...,English,1.0000,minimal_engagement,False,0,[],success,1952,42,1994


In [25]:
df[df["general_chat_id"]==13229]

,general_chat_id,total_messages,user_msg_count,bot_msg_count,start_time,end_time,conversation_duration,avg_response_time_user_sec,avg_response_time_ai_sec,max_gap_sec,...,conversation_language,language_confidence_score,conversation_status,has_questions,total_questions,qa_pairs,processing_status,input_tokens,output_tokens,total_tokens
12391,13229,6,3,3,2026-03-21 10:50:00,2026-03-21 10:57:20,0d 0h 7m 20s,220.0,0.0,371.0,...,English,0.9988,other,False,0,[],success,2833,40,2873


In [26]:
print(df.loc[12391,"full_conversation"])

User: أريد ترجمه هذه الرسالة الى الإنجليزية ،السلام عليكم ورحمه الله وبركاته  أود اخبارك يا اخى شئ اظن أنك لا تعلمه ،فى أول رساله ارسلتها لك بعرض الإسلام عليك اخبرتك ان تسمى سارة (وهذا اسم مرأه)ومن دولة مصر  ولكن هذا لم يكن المقصود أن أعرف من انت ولا من اين ولا المقصود أن اعلم أنك ذكر او انثى ولكن كان المقصود هو عرض الإسلام عليك بكل احترام وحريه الاختيار وهذا ما تم الحمد لله  ،ومن خلال محادثتى معك  لاحظت أنك اظن انى رجل وهذا ما أريد أن اخبارك به الآن بعد ان اطمئننت عليك فى الثبات على الإسلام والتعلم عنه  ،انى امرأه ،ومن الآن ان شاء الله ستنقطع المراسله بينى وبينك لأنك لم تعد بحاجه الى ذلك وانا اطمئننت عليك بشأن الإسلام ،ولكن لن ينقطع الدعاء لبعضنا بظهر الغيب وانا اوصيك بالثبات على الإسلام والتعلم عنه كثيرا حتى تستطيع الدعوه الى هذا الدين الحق وتعرف الناس عن نبينا الحبيب محمد صل الله عليه وسلم ،تعلم واثبت واصبر على المحن فلن تخلو هذه الحياه منها وننتظر حسن الختام  والجنه فى الاخره ،علم اخوانك عن الإسلام وتمسكوا ببعضكم وتعاونوا على البر والتقوى،تمسك بالقرآن والسنه لهذا الطريق الذى لا ضلا

In [10]:
print(df.loc[1,"qa_pairs"])

[]


In [ ]:
print(df.loc[1,"conversation_status"])